# CIFAR-10 — Experimentos complementares

A análise dos notebooks principais deixou duas perguntas em aberto. Este notebook as responde com o **mesmo protocolo**
(5 folds estratificados com a mesma seed, portanto as mesmas partições; escolha pela validação; teste revelado só no fim):

| Experimento | Lacuna identificada | O que muda |
|---|---|---|
| A — MLP: dropout com mais épocas | No Bloco 3, dropout 0,5 atingiu a melhor época perto do limite (42–48 de 50): o resultado pode ter sido cortado pelo orçamento. | dropout × função de erro com **100 épocas e paciência 10** |
| B — CNN: janela de pooling | No Bloco 3, com 4 blocos, só o pooling 2×2 era válido (3×3 colapsa o mapa; sem pooling = 270 M parâmetros). | blocos convolucionais × **janela de pooling** |

As receitas fixas são exatamente as dos campeões dos notebooks principais (escritas explicitamente nos grids, sem depender
dos resultados anteriores). A combinação 4 blocos + pooling 2×2 do Experimento B **reproduz o campeão da CNN** e serve de
checagem de reprodutibilidade.

**Tempo estimado:** ~40 min com 2× T4 (A: ~15 min; B: ~25 min).

**Antes de executar (Kaggle):** mesmas configurações dos notebooks principais — *GPU T4 ×2*, *Internet On*, Input
**cifar10-python** e secrets `GITHUB_TOKEN` / `WANDB_API_KEY` anexados. Execute com **Save Version → Save & Run All**.

## 0. Preparação do ambiente

In [ ]:
import os

REPO_URL = "https://github.com/diegoflyra/dfal-neural-networks.git"
REPO_DIR = "/tmp/dfal-neural-networks"  # fora de /kaggle/working: código e dataset não poluem o Output

# Repositório privado: crie o secret GITHUB_TOKEN (Add-ons → Secrets) com um token de leitura do GitHub.
# O token fica só em /tmp (não vai para o Output) e nunca é impresso.
clone_url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    clone_url = REPO_URL.replace("https://", "https://" + UserSecretsClient().get_secret("GITHUB_TOKEN") + "@")
    print("GitHub: usando GITHUB_TOKEN")
except Exception:
    print("GitHub: sem GITHUB_TOKEN (funciona apenas se o repositório for público)")

if os.path.isdir(REPO_DIR):
    !git -C $REPO_DIR pull -q
else:
    !git clone -q $clone_url $REPO_DIR
%cd $REPO_DIR
!git log -1 --oneline

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import shutil
import sys

import pandas as pd

WORKERS_PER_GPU = 2  # experimentos simultâneos por GPU
RESUME_FROM = ""  # ex.: "/kaggle/input/<output-da-versao-anterior>/outputs" para retomar

# CIFAR-10: usa a cópia anexada como Input do Kaggle (segundos), em vez do servidor original (lento).
# A cópia só é aceita se TODOS os arquivos tiverem o MD5 oficial (os mesmos hashes que o torchvision
# usa para validar o download de https://www.cs.toronto.edu/~kriz/cifar.html). Se algo divergir,
# a cópia é descartada e o dataset é baixado do servidor original.
import glob
import hashlib
import tarfile

from torchvision.datasets import CIFAR10

DATA_DIR = os.path.join(REPO_DIR, "data")
CIFAR_DIR = os.path.join(DATA_DIR, "cifar-10-batches-py")
OFFICIAL_MD5 = dict(CIFAR10.train_list + CIFAR10.test_list + [[CIFAR10.meta["filename"], CIFAR10.meta["md5"]]])
os.makedirs(DATA_DIR, exist_ok=True)


def md5(path):
    digest = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def cifar_is_official(folder):
    rows, ok = [], True
    for name, expected in OFFICIAL_MD5.items():
        path = os.path.join(folder, name)
        actual = md5(path) if os.path.isfile(path) else "arquivo ausente"
        ok &= actual == expected
        rows.append({"arquivo": name, "md5 oficial": expected, "md5 da cópia": actual,
                     "status": "OK" if actual == expected else "DIFERENTE"})
    display(pd.DataFrame(rows))
    return ok


if not os.path.isdir(CIFAR_DIR):
    folders = glob.glob("/kaggle/input/**/cifar-10-batches-py", recursive=True)
    archives = glob.glob("/kaggle/input/**/cifar-10-python.tar.gz", recursive=True)
    if folders:
        print(f"Cópia encontrada: {folders[0]}")
        shutil.copytree(folders[0], CIFAR_DIR)
    elif archives:
        print(f"Arquivo encontrado: {archives[0]} | md5 {md5(archives[0])} (oficial: {CIFAR10.tgz_md5})")
        with tarfile.open(archives[0]) as tar:
            tar.extractall(DATA_DIR)
    else:
        print("AVISO: CIFAR-10 não encontrado nos Inputs; será baixado do servidor original (pode levar muitos minutos).")

if os.path.isdir(CIFAR_DIR):
    if cifar_is_official(CIFAR_DIR):
        print("CIFAR-10 verificado: todos os arquivos são idênticos aos oficiais.")
    else:
        shutil.rmtree(CIFAR_DIR)
        print("CÓPIA REJEITADA: arquivos diferentes dos oficiais. O dataset será baixado do servidor original.")

# Onde os resultados são gravados (lido por run_experiment.py, grid_search.py e report_utils.py)
os.environ["EXP_OUTPUT_DIR"] = "/kaggle/working/outputs"
os.makedirs(os.environ["EXP_OUTPUT_DIR"], exist_ok=True)
if RESUME_FROM:
    shutil.copytree(RESUME_FROM, os.environ["EXP_OUTPUT_DIR"], dirs_exist_ok=True)
    print(f"Resultados anteriores copiados de {RESUME_FROM}; o que já foi concluído será pulado.")

# Weights & Biases via Kaggle Secrets; se falhar, os experimentos seguem apenas com o registro local.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
    print("W&B: chave carregada.")
except Exception as exc:
    os.environ["WANDB_MODE"] = "disabled"
    print(f"W&B desativado ({exc}). Resultados continuam em {os.environ['EXP_OUTPUT_DIR']}.")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
import report_utils as rep

In [ ]:
# Verificações rápidas antes de gastar GPU (sem treino):
# GPUs, flags → arquitetura, carregamento em GPU/K-fold (baixa o CIFAR-10) e construção de todas as configurações dos grids
!nvidia-smi -L
!python tests/check_hyperparams.py
!python tests/check_data_loader.py
!python tests/check_grids.py

## Experimento A — MLP: dropout e função de erro com orçamento maior

**Pergunta:** com mais épocas, dropout 0,5 alcança ou supera o dropout 0,2 do campeão? A vantagem da entropia cruzada
sobre MSE se mantém quando as duas têm tempo para convergir?

**Receita fixa (campeão da MLP):** 4 camadas × 256 neurônios, ReLU, Adam lr 3e-4, batch 128.
O dropout 0,2 com entropia cruzada é o próprio campeão, re-treinado no novo orçamento para uma comparação justa.

| Eixo | Valores |
|---|---|
| `dropout` | `0.2`, `0.5` |
| `loss_fn` | `cross_entropy`, `mse` |

**4 combinações.** Fixos no bloco: `mlp_layers=4`, `mlp_neurons=[256]`, `activation=relu`, `batch_norm=False`, `optimizer=adam`, `lr=0.0003`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=100`, `patience=10`, `eval_train=True`. Todas as configurações rodam os 5 folds.

**O que observar:** `melhor_epoca_media` (agora com folga até 100), a `val/accuracy` do dropout 0,5 em relação ao 0,2
e o `gap/accuracy`.

In [ ]:
!python src/grid_search.py grids/mlp_b3b_dropout_longo.json --workers_per_gpu 2

In [ ]:
rep.grid_ranking("mlp_b3b_dropout_longo", "final")

In [ ]:
rep.heatmap("mlp_b3b_dropout_longo", row="dropout", col="loss_fn")

In [ ]:
rep.heatmap("mlp_b3b_dropout_longo", row="dropout", col="loss_fn", value="melhor_epoca_media")

In [ ]:
rep.heatmap("mlp_b3b_dropout_longo", row="dropout", col="loss_fn", value="gap/accuracy_mean")

In [ ]:
rep.show_champion("mlp_b3b_dropout_longo")
rep.plot_finalists("mlp_b3b_dropout_longo")

### 📝 Análise — Experimento A

- **Dropout 0,5 precisou de quantas épocas? Alcançou o 0,2?** _…_
- **O resultado do Bloco 3 estava limitado pelo orçamento de épocas?** _…_
- **Entropia cruzada vs MSE com tempo suficiente:** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Experimento B — CNN: janela de max pooling

**Pergunta:** qual janela de pooling extrai melhor as características espaciais, e como isso interage com a profundidade?
Janelas maiores reduzem a resolução mais rápido (menos parâmetros e menos custo), mas descartam informação espacial
mais cedo.

**Receita fixa (campeão da CNN):** 64 filtros dobrando por bloco, kernel 3×3, padding `same`, stride 1, camada densa de 512,
BatchNorm, dropout 0,5, Adam lr 1e-3. Combinações cujo mapa espacial chega a 0×0 são descartadas antes do treino.

| Eixo | Valores |
|---|---|
| `conv_blocks` | `2`, `3`, `4` |
| `pool_size` | `2`, `3`, `4` |

**9 combinações.** Fixos no bloco: `filters=64`, `filters_growth=double`, `convs_per_block=1`, `kernel_size=3`, `padding=same`, `stride=1`, `fc_neurons=[512]`, `activation=relu`, `cnn_dropout=0.5`, `cnn_batch_norm=True`, `optimizer=adam`, `lr=0.001`, `momentum=0.9`, `weight_decay=0.0`, `batch_size=128`, `epochs=50`, `patience=7`, `eval_train=True`, `loss_fn=cross_entropy`. Todas as configurações rodam os 5 folds. Limite: 20,000,000 parâmetros.

**O que observar:** o heatmap blocos × pooling, o número de parâmetros de cada combinação e se a combinação
4 blocos + pooling 2×2 reproduz a `val/accuracy` do campeão do Bloco 3 (≈ 0,792).

In [ ]:
!python src/grid_search.py grids/cnn_b3b_pooling.json --workers_per_gpu 1

In [ ]:
rep.discarded_configs("cnn_b3b_pooling")

In [ ]:
rep.grid_ranking("cnn_b3b_pooling", "final")

In [ ]:
rep.heatmap("cnn_b3b_pooling", row="conv_blocks", col="pool_size")

In [ ]:
rep.heatmap("cnn_b3b_pooling", row="conv_blocks", col="pool_size", value="num_parameters")

In [ ]:
rep.heatmap("cnn_b3b_pooling", row="conv_blocks", col="pool_size", value="gap/accuracy_mean")

In [ ]:
rep.show_champion("cnn_b3b_pooling")
rep.plot_finalists("cnn_b3b_pooling")

### 📝 Análise — Experimento B

- **Qual janela de pooling foi melhor? Depende da profundidade?** _…_
- **Relação entre número de parâmetros e desempenho:** _…_
- **A combinação 4 blocos + pooling 2×2 reproduziu o campeão do Bloco 3?** _…_

In [ ]:
# Backup parcial: CSV/JSON/PNG/logs (os .pth ficam em outputs/ na aba Output)
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip

## Teste revelado

O teste é revelado apenas para a melhor configuração de cada experimento (escolhida pela validação), para comparação
com os campeões dos notebooks principais (MLP: 55,2% ± 0,3%; CNN: 78,3% ± 1,1% no teste).

In [ ]:
rep.final_report(["mlp_b3b_dropout_longo", "cnn_b3b_pooling"])

In [ ]:
!cd /kaggle/working && zip -qr outputs.zip outputs -x '*.pth' && ls -lh outputs.zip